# 01 — Data Inventory + Scientific QC (FIXED)

Builds a file inventory **and** checks raster CRS, bounds, resolution, NoData,
valid coverage and value range. It also flags likely duplicate precipitation datasets.

This notebook does not modify raw data.

In [ ]:
from pathlib import Path
import warnings

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists():
            return candidate
    raise FileNotFoundError(
        "Project root not found. Run this notebook from inside the repository."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

for d in [INTERIM_DIR, PROCESSED_DIR, OUTPUT_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
import hashlib, re
import numpy as np
import pandas as pd
import rasterio

def raster_quick_stats(path):
    row = {"path": str(path), "name": path.name, "size_mb": path.stat().st_size / 1024**2}
    try:
        with rasterio.open(path) as src:
            a = src.read(1, masked=True)
            valid = a.compressed()
            row.update({
                "crs": str(src.crs),
                "width": src.width,
                "height": src.height,
                "res_x": src.res[0],
                "res_y": src.res[1],
                "left": src.bounds.left,
                "bottom": src.bounds.bottom,
                "right": src.bounds.right,
                "top": src.bounds.top,
                "nodata": src.nodata,
                "dtype": src.dtypes[0],
                "valid_pct": 100.0 * valid.size / a.size if a.size else np.nan,
                "min": float(np.nanmin(valid)) if valid.size else np.nan,
                "max": float(np.nanmax(valid)) if valid.size else np.nan,
                "mean": float(np.nanmean(valid)) if valid.size else np.nan,
                "status": "OK",
            })
    except Exception as e:
        row["status"] = f"ERROR: {e}"
    return row

all_files = [p for p in RAW_DIR.rglob("*") if p.is_file()]
file_df = pd.DataFrame([{
    "path": str(p.relative_to(PROJECT_ROOT)),
    "folder": str(p.parent.relative_to(RAW_DIR)),
    "extension": p.suffix.lower(),
    "size_mb": p.stat().st_size / 1024**2,
} for p in all_files])

display(file_df.head())
print("\nFile counts by folder:")
display(file_df.groupby("folder").agg(files=("path","count"), total_mb=("size_mb","sum")).reset_index())

In [ ]:
rasters = sorted([*RAW_DIR.rglob("*.tif"), *RAW_DIR.rglob("*.tiff")])
raster_df = pd.DataFrame([raster_quick_stats(p) for p in rasters])
display(raster_df.head(20))

inventory_dir = PROCESSED_DIR / "inventory"
inventory_dir.mkdir(parents=True, exist_ok=True)
file_df.to_csv(inventory_dir / "raw_file_inventory.csv", index=False)
raster_df.to_csv(inventory_dir / "raw_raster_qc.csv", index=False)

bad = raster_df[raster_df["status"] != "OK"]
if len(bad):
    print("\nERROR rasters:")
    display(bad)
else:
    print("\nAll raster files are readable.")

In [ ]:
# Check expected monthly coverage for precipitation folders.
precip_root = RAW_DIR / "precipitation"
expected_months = {(y, m) for y in range(2017, 2023) for m in range(1, 13)}

def ym_from_name(name):
    # Supports 2017_01, 2017-01, 201701 and similar patterns.
    stem = Path(name).stem
    patterns = [
        r"(?<!\d)(20\d{2})[_-](0?[1-9]|1[0-2])(?!\d)",
        r"(?<!\d)(20\d{2})(0[1-9]|1[0-2])(?!\d)",
    ]
    for pat in patterns:
        m = re.search(pat, stem)
        if m:
            return int(m.group(1)), int(m.group(2))
    return None

coverage_rows = []
for folder in sorted([p for p in precip_root.iterdir() if p.is_dir()]):
    tif = sorted([*folder.rglob("*.tif"), *folder.rglob("*.tiff")])
    found = {x for p in tif if (x := ym_from_name(p.name))}
    missing = sorted(expected_months - found)
    coverage_rows.append({
        "product": folder.name,
        "tif_count": len(tif),
        "parsed_months": len(found),
        "missing_count": len(missing),
        "missing": str(missing[:20]),
    })

coverage = pd.DataFrame(coverage_rows)
display(coverage)
coverage.to_csv(inventory_dir / "precipitation_month_coverage.csv", index=False)

In [ ]:
# Duplicate check: especially CDR vs any extra PERSIANN folder.
# We compare corresponding monthly raster values after reading their native arrays
# only when shape/transform/CRS match exactly.
def compare_folders(folder_a, folder_b):
    A = RAW_DIR / "precipitation" / folder_a
    B = RAW_DIR / "precipitation" / folder_b
    if not A.exists() or not B.exists():
        return None
    amap = {ym_from_name(p.name): p for p in A.rglob("*.tif") if ym_from_name(p.name)}
    bmap = {ym_from_name(p.name): p for p in B.rglob("*.tif") if ym_from_name(p.name)}
    common = sorted(set(amap) & set(bmap))
    results = []
    for ym in common:
        with rasterio.open(amap[ym]) as a, rasterio.open(bmap[ym]) as b:
            same_grid = (
                a.width == b.width and a.height == b.height and
                a.transform == b.transform and a.crs == b.crs
            )
            same_values = False
            if same_grid:
                aa = a.read(1, masked=True)
                bb = b.read(1, masked=True)
                same_values = np.ma.allequal(aa, bb)
            results.append((ym, same_grid, same_values))
    return results

cmp = compare_folders("CDR", "PERSIANN")
if cmp is None:
    print("CDR/PERSIANN pair not both present — no comparison needed.")
else:
    n = len(cmp)
    same_grid = sum(x[1] for x in cmp)
    same_values = sum(x[2] for x in cmp)
    print(f"CDR vs PERSIANN common months: {n}")
    print(f"Same native grid: {same_grid}/{n}")
    print(f"Exactly same values: {same_values}/{n}")
    if n and same_values == n:
        print("WARNING: PERSIANN is an exact duplicate of CDR. Do NOT include it as a separate model feature.")
    else:
        print("Regardless of duplication, the corrected paper-style Comb1/Comb2 uses CDR as PERSIANN-CDR and excludes standalone PERSIANN.")

In [ ]:
print("\nInventory/QC files saved to:", inventory_dir)